# German retail market structure — national pipeline

One notebook, all of Germany, 2014–2025. It replaces `tdlinx_data.ipynb` (panel
build, ad-hoc HHI) and `catchment_analysis_bayern_2.ipynb` (Bayern catchments):
the panel front-end is section 1, the catchment solve is sections 2–4, and
everything after is new.

**What it produces**

| | |
|---|---|
| catchments | every populated 100 m cell assigned to its nearest store by OSRM drive time, per spec and year |
| cube | `shares` and `hhi` over spec × year × level × basis × actor level |
| figures | grid-assignment maps (Mannheim, Hamburg), choropleths, trends, distributions, agreement scatter |

**The dimension grid**

| dimension | values | n |
|---|---|---|
| metric | market share, HHI (+CR1/CR3/CR5) | 2 |
| year | 2014–2025 | 12 |
| level | Germany, Bundesland, Kreis | 3 |
| basis | catchment population, catchment area, VKFL, VKFL food, Umsatzpotential, Umsatzpotential food | 6 |
| spec | supermarkets; + discounters; + drugstores | 3 |
| actor | Gruppe (ownership), Chain (banner) | 2 |

Food/non-food multiplies only the two supply-side measures — a person allocated
to a store is not "food population" — so the basis dimension is 2×2 + 2 = 6, not 8.

**Cost.** The grid snap (section 3) is the expensive step and runs once for the
whole country; the travel-time matrices (section 4) run once per spec and resume
from checkpoints. Everything from section 6 on is seconds. Nothing is recomputed
if its artefact exists, so re-running the notebook end to end after a change to
the metrics is cheap.

**Logic lives in `scripts/run_pipeline.py`**, not in these cells, so the notebook
and the batch runner cannot drift:

```bash
python -m scripts.run_pipeline --stages all               # everything
python -m scripts.run_pipeline --stages cube,figures      # rebuild outputs only
python -m scripts.run_pipeline --stages all --land 09     # Bayern smoke test
```

## 0. Config

In [ ]:
import sys, gc, dataclasses
from pathlib import Path
import numpy as np, polars as pl, matplotlib.pyplot as plt

sys.path.append(str(Path().resolve().parent))

import scripts.run_pipeline as P
import scripts.isochrones.market_cube as C
import scripts.visualizations.figures as F
import scripts.isochrones.network_voronoi_2 as nv
import scripts.isochrones.market_metrics as M

# Every knob is in P.Config. Override with dataclasses.replace, never by editing
# the module, so the batch runner sees the same settings.
CFG = P.CFG
# CFG = dataclasses.replace(P.CFG, land="09")          # Bayern-only smoke test
# CFG = dataclasses.replace(P.CFG, years=(2024, 2025)) # short run

F.set_style()
print(CFG.griddir); print(CFG.outdir); print(P.SPECS)

ModuleNotFoundError: No module named 'scripts.isochrones.figures'

## 1. Panel + regions

`load_panel` filters to the union of all three specs and to `CFG.years`.
Kreis geometry is the Gemeinde layer dissolved on `ARS[:5]`; names come from the
Kreis layer, never from the dissolve. Kreis names repeat (München is 09162 *and*
09184), so `AGS5` is the only key and `LABEL` exists for titles and filenames.

In [ ]:
panel = P.load_panel(CFG)
kreis_gdf, bl_gdf = P.region_layers(CFG)
raster = P.clip_raster(CFG, kreis_gdf)          # national run: returns CFG.ghsl unchanged

print(panel.group_by("Shop Type").len().sort("len", descending=True))
print(kreis_gdf[["AGS5", "GEN", "LABEL"]].head())

panel: 447,303 rows | 45,072 stores | 2014-2025 | types ['Discount Geschäft', 'Drogeriemarkt', 'Supermarkt']
regions: 400 Kreise | 16 Bundesländer
shape: (3, 2)
┌───────────────────┬────────┐
│ Shop Type         ┆ len    │
│ ---               ┆ ---    │
│ str               ┆ u32    │
╞═══════════════════╪════════╡
│ Supermarkt        ┆ 197098 │
│ Discount Geschäft ┆ 190925 │
│ Drogeriemarkt     ┆ 59280  │
└───────────────────┴────────┘
    AGS5          GEN        LABEL
0  01001    Flensburg    Flensburg
1  01002         Kiel         Kiel
2  01003       Lübeck       Lübeck
3  01004   Neumünster   Neumünster
4  01051  Brunsbüttel  Brunsbüttel


## 2. Stores → snapped

Snapped **once**, for the union of all specs; each spec's store set is a filter of
this frame, so no store is ever snapped twice and the coordinates are identical
across specs.

`drop_over_m=300`: a real operating retail location has a road within 300 m.
Anything further is a geocode to check, and because a store is a candidate for
thousands of cells, one bad row corrupts a whole region.

In [ ]:
stores = P.stage_stores(panel, CFG)
print(stores.head())
print(stores.group_by("Shop Type").len())

prepare_stores: 3,483 store(s) have >1 distinct 'Gruppe' across the panel; most recent kept. Expected for banner switches.
prepare_stores: 1,121 store(s) have >1 distinct 'Shop Type' across the panel; most recent kept. Expected for banner switches.
prepare_stores: 1,830 store(s) move >100m across the panel; median position used. Inspect via coord_span_m.
prepare_stores: 447,303 panel rows -> 45,072 unique stores
snap: resuming from cache, 0/45,072 already snapped
  snap: 32/256 chunks
  snap: 64/256 chunks
  snap: 96/256 chunks
  snap: 128/256 chunks
  snap: 160/256 chunks
  snap: 192/256 chunks
  snap: 224/256 chunks
  snap: 256/256 chunks


PermissionError: [WinError 5] Access is denied: 'X:\\Project-Shares\\germart-lite\\germart-papaio\\edeka_project_2\\data\\catchments\\de_100m\\cache\\snap_45072_92def6a4.npz.tmp.npz' -> 'X:\\Project-Shares\\germart-lite\\germart-papaio\\edeka_project_2\\data\\catchments\\de_100m\\cache\\snap_45072_92def6a4.npz'

## 3. Cells → reduced + snapped  *(expensive, once)*

`adapt_resolution` picks each cell's resolution from its distance to the nearest
store, so using the **union** store set gives a grid at least as fine as any single
spec would have chosen — sharing it is conservative, and it pays the millions of
OSRM `/nearest` calls once instead of three times.

Order matters: stores are snapped before this, because resolution is chosen from
the distance to the nearest store and that distance should be measured to the
routable position. The mapping frames are written the moment they exist —
`group_by` order is not stable across runs, and regenerating them later yields
different node ids than the matrices already baked against them.

In [ ]:
cells, c2, mapping = P.stage_cells(stores, CFG, raster=raster)
TOTAL_POP = float(cells["pop"].cast(pl.Float64).sum())
print(f"{TOTAL_POP:,.0f} people on {cells.height:,} cells "
      f"({cells.height * 0.01:,.0f} km² inhabited)")

## 4. Node → region weights  *(once)*

Collapses the 35M-cell mapping to one row per (solve node, Kreis). A node that
straddles a border appears once per Kreis with its population split by the
constituent cells — which is why regions are labelled on the original grid and
never on the node.

Everything downstream joins against this instead of re-expanding 35M cells per
year, which is what makes 3 specs × 12 years cheap.

In [ ]:
node_reg = P.stage_regions(cells, mapping, kreis_gdf, CFG, raster=raster)
node_reg.head()

## 5. Per spec: travel times → assignments  *(expensive, resumable)*

One matrix per spec. The matrix is year-invariant (it holds every store that ever
appears), so the 12 yearly assignments cost nothing extra: `assign` filters to the
stores alive in that year.

`refine_until` escalates *k* until `CERT_TARGET` of nodes are certified at rank 1,
re-solving only the nodes still uncertified at each step. An uncertified node is
one where a store outside the candidate set *could* be nearer, so its catchment is
unproven — certification is not a formality.

Kill this cell and re-run it: `travel_times` and `refine_until` both checkpoint.

In [ ]:
solved = {}
for spec in P.SPECS:
    st = P.spec_stores(stores, spec)
    print(f"\n=== {spec}: {st.height:,} stores ===")
    cand, dur = P.stage_matrix(spec, c2, st, CFG)
    assigns  = P.stage_assign(spec, c2, st, cand, dur, CFG,
                              cells=cells, mapping=mapping, raster=raster)
    C.coverage(assigns[max(assigns)], node_reg, TOTAL_POP)
    solved[spec] = dict(stores=st, assignments=assigns, cand=cand, dur=dur)
    gc.collect()

## 6. The cube

Two tidy tables, and nothing downstream reads anything else:

* `shares` — one row per (spec, year, level, region, basis, actor_level, actor)
* `hhi` — one row per (spec, year, level, region, basis, actor_level), with CR1/CR3/CR5

`validate` raises if shares fail to sum to 1 in any cell, if HHI or CR leave
their range, or if CR1 exceeds CR3.

In [ ]:
for spec, s in solved.items():
    P.stage_cube(spec, s["assignments"], node_reg, panel, s["stores"], CFG)

shares, hhi = P.read_cube(CFG)
C.validate(shares, hhi)
print(shares.height, "share rows |", hhi.height, "hhi rows")
shares.head()

## 7. Headline tables

`headline` is one row per (spec, year) for a chosen slice. Read the population
basis against the VKFL basis: where they disagree, the catchment crosses an
administrative border, and that gap is the first thing a referee asks about.

In [ ]:
YEAR = int(hhi["year"].max())

print("National HHI by spec and year — shares on catchment population, Gruppe level")
print(C.headline(hhi, level="DE", basis="pop", actor_level="Gruppe")
      .pivot(values="hhi", index="year", on="spec").sort("year"))

print("\nSame, shares on sales area (VKFL)")
print(C.headline(hhi, level="DE", basis="vkfl", actor_level="Gruppe")
      .pivot(values="hhi", index="year", on="spec").sort("year"))

print(f"\nTop Gruppen {YEAR}, supermarkets + discounters, every basis")
print(shares.filter(pl.col("spec") == "super_disc", pl.col("year") == YEAR,
                    pl.col("level") == "DE", pl.col("actor_level") == "Gruppe",
                    pl.col("rank") <= 8)
      .pivot(values="share", index="actor", on="basis")
      .sort("pop", descending=True))

In [ ]:
# Kreis-level dispersion: the national line hides it. An unchanged national HHI is
# compatible with half the Kreise concentrating and half loosening.
kr = hhi.filter(pl.col("level") == "KREIS", pl.col("actor_level") == "Gruppe",
                pl.col("basis") == "pop", pl.col("region") != "UNKNOWN")
print(kr.group_by("spec", "year").agg(
        pl.col("hhi").median().alias("median"),
        pl.col("hhi").quantile(0.1).alias("p10"),
        pl.col("hhi").quantile(0.9).alias("p90"),
        (pl.col("hhi") > 2500).mean().alias("share_above_2500"),
        pl.col("n_actors").mean().alias("mean_gruppen"))
      .sort("spec", "year"))

print("\nMost concentrated Kreise,", YEAR)
names = dict(zip(kreis_gdf["AGS5"].astype(str), kreis_gdf["LABEL"].astype(str)))
print(kr.filter(pl.col("year") == YEAR, pl.col("spec") == "super_disc")
        .sort("hhi", descending=True).head(12)
        .with_columns(pl.col("region").replace(names).alias("Kreis"))
        .select("Kreis", "region", "hhi", "cr1", "cr3", "n_actors", "n_stores", "total"))

## 8. Grid assignment maps — Mannheim and Hamburg

The catchment solve pushed back down to the original 100 m cells: one square per
raster cell, coloured by the Gruppe that wins it, white seams where the winning
*store* changes. Where the solve was coarsened, identical squares appear in
blocks — that blockiness is the honest picture of the reduction, not an artefact
of the drawing.

The `zoom` variant is a 6 km window with cell gridlines, so the squares themselves
are legible; the `city` variant covers the Kreis plus 3 km of surroundings.

In [ ]:
grid_files = []
for spec in P.SPECS:
    grid_files += P.stage_grid_maps(spec, cells, mapping, solved[spec]["assignments"],
                                    solved[spec]["stores"], kreis_gdf, CFG, year=YEAR)

# and inline, one at a time
a = solved["super_disc"]["assignments"][YEAR]
st = solved["super_disc"]["stores"]
for city, ags5, half in (("Mannheim", "08222", 3.0), ("Hamburg", "02000", 3.0)):
    bbox = F.centre_bbox(kreis_gdf, "AGS5", ags5, half_km=half)
    g = F.grid_slice(cells, mapping, a, st, bbox=bbox, by="Gruppe")
    F.plot_grid(g, stores=st, scale_km=1.0,
                title=f"{city} — 100 m grid, nearest store by drive time, {YEAR}")
    plt.show()

## 9. Figures

Every panel is a slice of the cube, so any number in any figure traces back to
rows in `shares` / `hhi`. `F.save` writes PDF (vector, for the paper) and PNG
(300 dpi, for slides) from one call — no figure exists in only one format.

In [ ]:
files = P.stage_figures(shares, hhi, kreis_gdf, bl_gdf, CFG, year=YEAR)
print(len(files), "files")

# one inline example of each type
F.plot_choropleth(hhi, kreis_gdf, level="KREIS", spec="super_disc", year=YEAR,
                  basis="pop", actor_level="Gruppe", label="HHI (0–10,000)",
                  title=f"HHI by Kreis, supermarkets + discounters, {YEAR}"); plt.show()
F.plot_trend(hhi, basis="pop", actor_level="Gruppe", by="spec",
             title="National HHI, catchment population"); plt.show()
F.plot_share_trend(shares, level="DE", region="DE", spec="super_disc", basis="pop",
                   actor_level="Gruppe", top=6, title="National Gruppe shares"); plt.show()
F.plot_basis_agreement(hhi, spec="super_disc", year=YEAR, actor_level="Gruppe",
                       title=f"Demand- vs supply-side HHI by Kreis, {YEAR}"); plt.show()

## 10. Robustness

The headline is winner-take-all, so there is no elasticity to vary. What remains
is sensitivity to the **market definition**: how wide the local isochrone is, and
whether shares are read off population or sales area.

Probabilistic (Huff) allocation is parked in `huff.py`: `beta` is assumed rather
than estimated and the choice set is truncated by *k*, which biases every retained
store's share upward. When it is ready it returns the same `(cell_id, store_id,
share)` frame and drops in with one line changed.

In [ ]:
spec = "super_disc"
st, cand, dur = solved[spec]["stores"], solved[spec]["cand"], solved[spec]["dur"]
sty = st.join(nv.store_year_attrs(panel, YEAR, ["VKFL"]), on="store_id",
              how="left").sort("store_id")

rows = []
for mins in (10, 15, 20, 25, 30):
    iso = nv.assign(c2, st, cand, dur, year=YEAR, ranks=cand.k, max_seconds=mins * 60)
    lm  = M.local_market(iso, sty, by="Gruppe",
                         catchment=solved[spec]["assignments"][YEAR])
    rows.append({"isochrone_min": mins,
                 "local_hhi_median": lm["local_hhi"].median(),
                 "mean_gruppen": lm["n_groups"].mean(),
                 "mean_stores": lm["n_stores"].mean(),
                 "monopoly_share": lm.filter(pl.col("local_hhi") >= 9_999).height / lm.height})
print(pl.DataFrame(rows))

In [ ]:
# Basis sensitivity across the whole Kreis distribution, all specs at once.
piv = (hhi.filter(pl.col("level") == "KREIS", pl.col("year") == YEAR,
                  pl.col("actor_level") == "Gruppe", pl.col("region") != "UNKNOWN")
       .group_by("spec", "basis").agg(pl.col("hhi").median().alias("median_kreis_hhi"))
       .pivot(values="median_kreis_hhi", index="spec", on="basis"))
print(piv)

# Gruppe vs Chain: the gap is what cooperative ownership adds to measured concentration.
print(hhi.filter(pl.col("level") == "DE", pl.col("year") == YEAR, pl.col("basis") == "pop")
        .select("spec", "actor_level", "hhi", "cr1", "cr3", "n_actors")
        .sort("spec", "actor_level"))

## 11. Export

Parquet is the archive (`outputs/.../cube/`); CSV is what goes to co-authors and
into the report. One file per level keeps the Kreis file openable in Excel.

In [ ]:
exp = CFG.outdir / "cube" / "csv"; exp.mkdir(parents=True, exist_ok=True)
names_bl = dict(zip(bl_gdf["AGS2"].astype(str), bl_gdf["LABEL"].astype(str)))
lab = {**names, **names_bl, "DE": "Deutschland"}

for lev in ("DE", "BL", "KREIS"):
    (hhi.filter(pl.col("level") == lev)
        .with_columns(pl.col("region").replace(lab).alias("region_name"))
        .sort("spec", "year", "basis", "actor_level", "region")
        .write_csv(exp / f"hhi_{lev.lower()}.csv"))
    (shares.filter(pl.col("level") == lev)
        .with_columns(pl.col("region").replace(lab).alias("region_name"))
        .sort("spec", "year", "basis", "actor_level", "region", "rank")
        .write_csv(exp / f"shares_{lev.lower()}.csv"))
print(sorted(p.name for p in exp.iterdir()))

---

### Rebuilding from scratch

```bash
python -m scripts.run_pipeline --stages all                  # respects every cache
python -m scripts.run_pipeline --stages cube,figures --force cube,figures
```

Delete an artefact to force its stage; `--force <stage>` does the same without
deleting. If you change a reduction or candidate parameter, bump `Config.grid`
(new grid namespace) or delete the matrices — `cells`, `mapping` and `matrix` are
one object split across three files, and mixing versions produces wrong answers
rather than errors.

Full documentation: `docs/market_structure.md`.